# Appendix E — Sensitivity to the large-pool query budget

Compare budgets of 100, 250, 500, 750 and 1,000 queries. Keep the pool, seeds,
classifier, thresholds and discovery fraction fixed. Reuse trajectories where
the protocol permits, and distinguish fresh lower-budget runs from points
read from a longer trajectory.


## 1. Experimental choice


In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, time, warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='.*sklearn\\.utils\\.parallel\\.delayed.*', category=UserWarning)
warnings.filterwarnings('ignore')

raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
assert (raiz / 'config.py').exists(), 'Open this notebook inside the project directory'
os.chdir(raiz)
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import config
from src import pipeline
from src.models import active_learning as al
from src.models import anomalies as an
from src.models import classification, evaluation

RS = config.RANDOM_STATE
plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 200, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
})
COLORES = {'Two channels': '#2166ac', 'Triage': '#d6604d', 'Random': '#9aa0a6'}


In [ ]:
BATCH_COMB = 99
PRESUPUESTOS = (100, 250, 500, 750, 1000)
PRESUPUESTOS_NUEVOS = (250, 750)
B_ITER = 100
N_ITER = 30
FRAC_DESC = 0.6
SEEDS = tuple(range(10))
TEST_SIZE_EVAL = 0.2
RECOMPUTE = False

CSV_DC1000 = config.DIR_PROCESSED / 'dos_canales_batch99_10seeds.csv'
CSV_TRI1000 = config.DIR_PROCESSED / 'triaje_batch99_10seeds.csv'
CSV_DC = {
    b: config.DIR_PROCESSED / f'dos_canales_batch99_budget{b}_10seeds.csv'
    for b in PRESUPUESTOS_NUEVOS
}
CSV_TRI = {
    b: config.DIR_PROCESSED / f'triaje_batch99_budget{b}_10seeds.csv'
    for b in PRESUPUESTOS_NUEVOS
}
CSV_COMPARACION = (
    config.DIR_PROCESSED /
    'comparacion_batch99_presupuestos_100_250_500_750_1000_10seeds.csv'
)
CSV_PAREADA = (
    config.DIR_PROCESSED /
    'comparacion_pareada_batch99_presupuestos_100_250_500_750_1000.csv'
)

assert CSV_DC1000.exists() and CSV_TRI1000.exists()
print('Presupuestos:', PRESUPUESTOS)
print('Maximum queries per round:', B_ITER)
print('Semillas:', SEEDS)


## 2. Check caches and prepare the pool


In [ ]:
def cache_dc_valida(file_path, presupuesto):
    if not file_path.exists():
        return False
    d = pd.read_csv(file_path)
    if set(d.get('semilla', [])) != set(SEEDS):
        return False
    if set(np.round(d.get('frac_desc', []), 8)) != {FRAC_DESC}:
        return False
    fin = d.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
    return len(fin) == len(SEEDS) and (fin['n_consultas'] == presupuesto).all()

def cache_tri_valida(file_path, presupuesto):
    if not file_path.exists():
        return False
    d = pd.read_csv(file_path)
    if set(d.get('semilla', [])) != set(SEEDS):
        return False
    if set(d.get('modo', [])) != {'triage', 'random'}:
        return False
    fin = d.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
    return len(fin) == 2 * len(SEEDS) and (fin['n_consultas'] == presupuesto).all()

NECESITA = {
    b: RECOMPUTE or not (
        cache_dc_valida(CSV_DC[b], b) and cache_tri_valida(CSV_TRI[b], b)
    )
    for b in PRESUPUESTOS_NUEVOS
}
print('Ejecuciones necesarias:', NECESITA)

if not RECOMPUTE and any(NECESITA.values()):
    raise FileNotFoundError('Valid budget trajectories are missing. Import saved results or explicitly set RECOMPUTE=True to repeat AL.')


In [ ]:
if any(NECESITA.values()):
    tr = classification.features_e2gp('train')
    X_tr, COLS = classification.e2gp_matrix(tr)
    y_tr = tr['target'].to_numpy()

    batch_paths = pipeline.select_batch(BATCH_COMB)
    assert batch_paths.features_gp2d_test_csv.exists(), (
        'Combined-pool features are missing; run notebook 07 first.'
    )
    te = classification.features_e2gp('test', batch=BATCH_COMB)
    metadata_table = pd.read_csv(batch_paths.test_metadata_clean_csv)
    te = te.merge(metadata_table[['object_id', 'true_target']], on='object_id')
    tt = te['true_target'].to_numpy()
    X_te, _ = classification.e2gp_matrix(te, COLS)

    clf0 = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
    gp_te = te[['object_id']].merge(
        pd.read_csv(batch_paths.features_gp_test_csv), on='object_id', how='left'
    )
    st_te = te[['object_id']].merge(
        pd.read_csv(batch_paths.features_test_csv), on='object_id', how='left'
    )
    S = np.column_stack([
        an.confidence_novelty_score(clf0.predict_proba(X_te)),
        an.score_isolation_forest(X_te),
        an.gp_residual_score(gp_te),
        an.amplitude_score(st_te),
    ])
    desc_te = S[:, 1:]

    known_objects = tt < 990
    valores, class_counts = np.unique(tt[known_objects], return_counts=True)
    unitarias = set(valores[class_counts < 2].tolist())
    keep = ~(known_objects & np.isin(tt, list(unitarias)))
    X_te, tt, desc_te = X_te[keep], tt[keep], desc_te[keep]

    def make_model(y_lab):
        return classification.e2gp_al(y=y_lab, random_state=RS)

    print(f'Pool preparado: {len(tt):,} objects; '
          f'{int((tt >= 990).sum()):,} anomalies 99x.')
else:
    print('All additional runs are already available.')


## 3. Fresh runs with 250 and 750 queries


In [ ]:
hist_dc = {}
hist_tri = {}

for presupuesto in PRESUPUESTOS_NUEVOS:
    if NECESITA[presupuesto]:
        inicio = time.time()
        hist_dc[presupuesto] = al.two_channel_multiseed(
            X_tr, y_tr, X_te, tt, desc_te, make_model,
            fracs=(FRAC_DESC,), random_seeds=SEEDS,
            test_size_eval=TEST_SIZE_EVAL,
            conf_dudoso=0.60, reg_C=0.1,
            budget=presupuesto, b_iter=B_ITER, n_iter=N_ITER,
            output_path=str(CSV_DC[presupuesto]), verbose=True,
        )
        print(f'Two channels B={presupuesto}: {(time.time()-inicio)/60:.1f} min.')

        inicio = time.time()
        hist_tri[presupuesto] = al.confidence_triage_multiseed(
            X_tr, y_tr, X_te, tt, make_model,
            modes_list=('triage', 'random'), random_seeds=SEEDS,
            test_size_eval=TEST_SIZE_EVAL,
            conf_anom=0.30, conf_dudoso=0.60, orden_dudoso='entropy',
            budget=presupuesto, b_iter=B_ITER, n_iter=N_ITER,
            output_path=str(CSV_TRI[presupuesto]), verbose=True,
        )
        print(f'Triage+random B={presupuesto}: {(time.time()-inicio)/60:.1f} min.')
    else:
        hist_dc[presupuesto] = pd.read_csv(CSV_DC[presupuesto])
        hist_tri[presupuesto] = pd.read_csv(CSV_TRI[presupuesto])

    assert cache_dc_valida(CSV_DC[presupuesto], presupuesto)
    assert cache_tri_valida(CSV_TRI[presupuesto], presupuesto)

    fin_dc = (hist_dc[presupuesto].sort_values('iter')
              .groupby(['frac_desc', 'semilla']).tail(1))
    fin_tri = (hist_tri[presupuesto].sort_values('iter')
               .groupby(['modo', 'semilla']).tail(1))
    rondas_esperadas = int(np.ceil(presupuesto / B_ITER))
    assert (fin_dc['iter'] == rondas_esperadas).all()
    assert (fin_tri['iter'] == rondas_esperadas).all()
    print(f'B={presupuesto}: integridad verificada ({rondas_esperadas} rondas).')


## 4. Assemble the budget curve


In [ ]:
hist_dc1000 = pd.read_csv(CSV_DC1000)
hist_tri1000 = pd.read_csv(CSV_TRI1000)
hist_dc1000 = hist_dc1000[np.isclose(hist_dc1000['frac_desc'], FRAC_DESC)]

def extraer_final_exacto(hist, claves, presupuesto):
    d = hist[hist['n_consultas'] == presupuesto].copy()
    if len(d) == 0:
        raise ValueError(f'No trajectory point exists at q={presupuesto}')
    fin = d.sort_values('iter').groupby(claves).tail(1)
    return fin

finales_dc = {}
finales_tri = {}
for presupuesto in PRESUPUESTOS:
    if presupuesto in PRESUPUESTOS_NUEVOS:
        finales_dc[presupuesto] = (
            hist_dc[presupuesto].sort_values('iter')
            .groupby(['frac_desc', 'semilla']).tail(1)
        )
        finales_tri[presupuesto] = (
            hist_tri[presupuesto].sort_values('iter')
            .groupby(['modo', 'semilla']).tail(1)
        )
    else:
        finales_dc[presupuesto] = extraer_final_exacto(
            hist_dc1000, ['frac_desc', 'semilla'], presupuesto
        )
        finales_tri[presupuesto] = extraer_final_exacto(
            hist_tri1000, ['modo', 'semilla'], presupuesto
        )

def finales_largos(dc, tri, presupuesto):
    piezas = [
        dc.assign(sistema='Two channels', presupuesto=presupuesto),
        tri[tri['modo'] == 'triage'].assign(sistema='Triage', presupuesto=presupuesto),
        tri[tri['modo'] == 'random'].assign(sistema='Random', presupuesto=presupuesto),
    ]
    feature_columns = ['presupuesto', 'sistema', 'semilla', 'n_consultas',
                'MF1', 'recall_anom', 'n_descubiertas', 'n_auto']
    return pd.concat(piezas, ignore_index=True)[feature_columns]

finales = pd.concat([
    finales_largos(finales_dc[b], finales_tri[b], b)
    for b in PRESUPUESTOS
], ignore_index=True)
finales.to_csv(CSV_COMPARACION, index=False)

summary_table = (finales.groupby(['presupuesto', 'sistema'])
           .agg(
               consultas=('n_consultas', 'mean'),
               MF1_media=('MF1', 'mean'),
               MF1_sd=('MF1', 'std'),
               recall_media=('recall_anom', 'mean'),
               recall_sd=('recall_anom', 'std'),
               hallazgos_media=('n_descubiertas', 'mean'),
               hallazgos_sd=('n_descubiertas', 'std'),
               auto_media=('n_auto', 'mean'),
           ).reset_index())
for col in ['MF1_sd', 'recall_sd', 'hallazgos_sd']:
    summary_table[col.replace('_sd', '_ee')] = summary_table[col] / np.sqrt(len(SEEDS))
display(summary_table.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
configuraciones = [
    ('hallazgos_media', 'hallazgos_sd', '99x discoveries', 'Cumulative anomaly discoveries'),
    ('MF1_media', 'MF1_sd', 'Macro-F1', 'Final classification'),
]
for ax, (columna, columna_sd, ylabel, plot_title) in zip(axes, configuraciones):
    for sistema in ['Two channels', 'Triage', 'Random']:
        d = summary_table[summary_table['sistema'] == sistema].sort_values('presupuesto')
        error = d[columna_sd]
        error_inferior = np.minimum(error.to_numpy(), d[columna].to_numpy())
        error_asimetrico = np.vstack([error_inferior, error.to_numpy()])
        ax.errorbar(
            d['presupuesto'], d[columna], yerr=error_asimetrico,
            fmt='-o', color=COLORES[sistema], lw=2, ms=6,
            capsize=3, elinewidth=1.2, alpha=0.95, label=sistema,
        )
    ax.set(xlabel='Total budget', ylabel=ylabel, title=plot_title)
    ax.set_xticks(PRESUPUESTOS)
    ax.legend()
fig.suptitle('Budget sensitivity in the combined pool · mean ± SD · 10 seeds')
fig.tight_layout()
ruta_figura = config.FIGURE_DIR / 'sensibilidad_presupuesto_escala_100_1000.png'
fig.savefig(ruta_figura, bbox_inches='tight')
plt.show()
print(ruta_figura)


## 5. Paired comparisons of two channels and triage


In [ ]:
pareadas = []
for presupuesto in PRESUPUESTOS:
    tri = finales_tri[presupuesto]
    tri = tri[tri['modo'] == 'triage']
    comp = evaluation.paired_comparison_summary(
        finales_dc[presupuesto], tri,
        ['MF1', 'recall_anom'], n_boot=20_000, seed_value=RS,
    )
    comp.insert(0, 'presupuesto', presupuesto)
    pareadas.append(comp)
pareadas = pd.concat(pareadas, ignore_index=True)
pareadas.to_csv(CSV_PAREADA, index=False)
display(pareadas.round(5))


## 6. Automatically computed summaries

Use the computed contrasts rather than a prewritten significance claim.
Different budgets on the same pool are correlated sensitivity checks and do
not provide independent replications.


In [ ]:
tabla_h = summary_table.pivot(index='presupuesto', columns='sistema',
                         values='hallazgos_media').loc[list(PRESUPUESTOS)]
tabla_f = summary_table.pivot(index='presupuesto', columns='sistema',
                         values='MF1_media').loc[list(PRESUPUESTOS)]

lineas = [
    '### Numerical summary',
    '',
    '| Presupuesto | Hallazgos two channels | Hallazgos triage | Hallazgos azar | '
    'MF1 two channels | MF1 triage | MF1 azar |',
    '|---:|---:|---:|---:|---:|---:|---:|',
]
for b in PRESUPUESTOS:
    lineas.append(
        f"| {b} | {tabla_h.loc[b, 'Two channels']:.1f} | "
        f"{tabla_h.loc[b, 'Triage']:.1f} | {tabla_h.loc[b, 'Random']:.1f} | "
        f"{tabla_f.loc[b, 'Two channels']:.3f} | "
        f"{tabla_f.loc[b, 'Triage']:.3f} | {tabla_f.loc[b, 'Random']:.3f} |"
    )
lineas += [
    '',
    'The budgets are experimental label counts rather than calibrated spectroscopic exposure times. One hundred queries measure only the first round; later points incorporate acquired feedback.',
]
display(Markdown('\n'.join(lineas)))


## 7. Interpretation

Evaluate when the discovery advantage develops after feedback and what
classification cost accompanies it. These label counts are experimental
budgets, not an instrument-specific conversion to spectroscopic exposure time.
